# Transformer-Based Audio Generation
## Deep Learning for Amapiano Music Generation

This notebook implements a transformer-based model for audio generation, inspired by MusicGen and AudioLM architectures.

In [ ]:
!pip install -q torch torchaudio transformers accelerate einops

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchaudio
import numpy as np
import math
from einops import rearrange
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Audio Tokenizer (EnCodec-style)

Convert audio waveforms to discrete tokens for transformer processing.

In [ ]:
class AudioEncoder(nn.Module):
    """Convolutional encoder for audio."""
    
    def __init__(self, channels=32, latent_dim=128):
        super().__init__()
        
        self.encoder = nn.Sequential(
            # Downsample by 320x total
            nn.Conv1d(1, channels, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.Conv1d(channels, channels * 2, kernel_size=7, stride=4, padding=3),
            nn.ReLU(),
            nn.Conv1d(channels * 2, channels * 4, kernel_size=7, stride=5, padding=3),
            nn.ReLU(),
            nn.Conv1d(channels * 4, channels * 8, kernel_size=7, stride=8, padding=3),
            nn.ReLU(),
            nn.Conv1d(channels * 8, latent_dim, kernel_size=3, stride=1, padding=1),
        )
    
    def forward(self, x):
        # x: (batch, samples)
        x = x.unsqueeze(1)  # (batch, 1, samples)
        return self.encoder(x)  # (batch, latent_dim, frames)


class VectorQuantizer(nn.Module):
    """Vector Quantization layer."""
    
    def __init__(self, n_embeddings=1024, embedding_dim=128, commitment_cost=0.25):
        super().__init__()
        
        self.n_embeddings = n_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        
        self.embeddings = nn.Embedding(n_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1/n_embeddings, 1/n_embeddings)
    
    def forward(self, z):
        # z: (batch, dim, frames)
        z = z.permute(0, 2, 1).contiguous()  # (batch, frames, dim)
        z_flat = z.view(-1, self.embedding_dim)
        
        # Calculate distances
        distances = torch.cdist(z_flat, self.embeddings.weight)
        
        # Find nearest embedding
        encoding_indices = distances.argmin(dim=1)
        
        # Quantize
        z_q = self.embeddings(encoding_indices).view(z.shape)
        
        # Loss
        loss = F.mse_loss(z_q.detach(), z) + self.commitment_cost * F.mse_loss(z_q, z.detach())
        
        # Straight-through estimator
        z_q = z + (z_q - z).detach()
        
        z_q = z_q.permute(0, 2, 1).contiguous()  # (batch, dim, frames)
        encoding_indices = encoding_indices.view(z.shape[0], z.shape[1])
        
        return z_q, loss, encoding_indices


class AudioDecoder(nn.Module):
    """Convolutional decoder for audio."""
    
    def __init__(self, channels=32, latent_dim=128):
        super().__init__()
        
        self.decoder = nn.Sequential(
            nn.Conv1d(latent_dim, channels * 8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(channels * 8, channels * 4, kernel_size=16, stride=8, padding=4),
            nn.ReLU(),
            nn.ConvTranspose1d(channels * 4, channels * 2, kernel_size=10, stride=5, padding=2),
            nn.ReLU(),
            nn.ConvTranspose1d(channels * 2, channels, kernel_size=8, stride=4, padding=2),
            nn.ReLU(),
            nn.ConvTranspose1d(channels, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )
    
    def forward(self, z):
        return self.decoder(z).squeeze(1)


class AudioTokenizer(nn.Module):
    """Complete audio tokenizer with encoder, VQ, and decoder."""
    
    def __init__(self, n_embeddings=1024, embedding_dim=128):
        super().__init__()
        
        self.encoder = AudioEncoder(latent_dim=embedding_dim)
        self.vq = VectorQuantizer(n_embeddings, embedding_dim)
        self.decoder = AudioDecoder(latent_dim=embedding_dim)
    
    def encode(self, x):
        z = self.encoder(x)
        _, _, indices = self.vq(z)
        return indices
    
    def decode(self, indices):
        z_q = self.vq.embeddings(indices).permute(0, 2, 1)
        return self.decoder(z_q)
    
    def forward(self, x):
        z = self.encoder(x)
        z_q, vq_loss, indices = self.vq(z)
        x_recon = self.decoder(z_q)
        
        # Reconstruction loss
        recon_loss = F.mse_loss(x_recon, x[:, :x_recon.shape[1]])
        
        return x_recon, recon_loss + vq_loss, indices

# Test tokenizer
tokenizer = AudioTokenizer(n_embeddings=1024).to(device)
test_audio = torch.randn(2, 22050 * 5).to(device)
recon, loss, indices = tokenizer(test_audio)
print(f"Input shape: {test_audio.shape}")
print(f"Token indices shape: {indices.shape}")
print(f"Reconstructed shape: {recon.shape}")

## 2. Transformer Language Model for Audio Tokens

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""
    
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class AudioTransformer(nn.Module):
    """Transformer for audio token generation."""
    
    def __init__(self, vocab_size=1024, d_model=512, nhead=8, 
                 num_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.d_model = d_model
        
        # Token embedding
        self.token_embedding = nn.Embedding(vocab_size + 1, d_model)  # +1 for start token
        self.pos_encoding = PositionalEncoding(d_model)
        
        # Transformer decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers)
        
        # Output projection
        self.output_proj = nn.Linear(d_model, vocab_size)
        
        # Conditioning projection (for text/tag conditioning)
        self.condition_proj = nn.Linear(d_model, d_model)
    
    def generate_causal_mask(self, seq_len, device):
        """Generate causal attention mask."""
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        return mask.bool()
    
    def forward(self, tokens, condition=None):
        # tokens: (batch, seq_len)
        batch_size, seq_len = tokens.shape
        
        # Embed tokens
        x = self.token_embedding(tokens)
        x = self.pos_encoding(x)
        
        # Generate causal mask
        causal_mask = self.generate_causal_mask(seq_len, tokens.device)
        
        # Conditioning (if provided)
        if condition is not None:
            memory = self.condition_proj(condition).unsqueeze(1)
        else:
            memory = torch.zeros(batch_size, 1, self.d_model, device=tokens.device)
        
        # Transformer
        x = self.transformer(x, memory, tgt_mask=causal_mask)
        
        # Output logits
        logits = self.output_proj(x)
        
        return logits
    
    @torch.no_grad()
    def generate(self, start_tokens, max_length=500, temperature=1.0, top_k=50):
        """Generate audio tokens autoregressively."""
        self.eval()
        
        tokens = start_tokens.clone()
        
        for _ in tqdm(range(max_length), desc='Generating'):
            logits = self(tokens)
            logits = logits[:, -1, :] / temperature
            
            # Top-k sampling
            if top_k > 0:
                indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
                logits[indices_to_remove] = float('-inf')
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            tokens = torch.cat([tokens, next_token], dim=1)
        
        return tokens

# Test transformer
transformer = AudioTransformer(vocab_size=1024).to(device)
test_tokens = torch.randint(0, 1024, (2, 100)).to(device)
logits = transformer(test_tokens)
print(f"Input tokens shape: {test_tokens.shape}")
print(f"Output logits shape: {logits.shape}")
print(f"Total parameters: {sum(p.numel() for p in transformer.parameters()):,}")

## 3. Amapiano-Specific Conditioning

In [ ]:
class AmapianoConditioner(nn.Module):
    """Conditioning module for Amapiano-specific generation."""
    
    def __init__(self, d_model=512):
        super().__init__()
        
        # Genre embedding
        self.genre_embedding = nn.Embedding(10, d_model // 4)  # 10 Amapiano sub-genres
        
        # BPM embedding (binned: 100-130 BPM in 5 BPM bins)
        self.bpm_embedding = nn.Embedding(7, d_model // 4)
        
        # Energy level (low, medium, high)
        self.energy_embedding = nn.Embedding(3, d_model // 4)
        
        # Region style (Johannesburg, Pretoria, Durban, Cape Town)
        self.region_embedding = nn.Embedding(4, d_model // 4)
        
        # Combine embeddings
        self.combine = nn.Linear(d_model, d_model)
    
    def forward(self, genre_idx, bpm_idx, energy_idx, region_idx):
        genre = self.genre_embedding(genre_idx)
        bpm = self.bpm_embedding(bpm_idx)
        energy = self.energy_embedding(energy_idx)
        region = self.region_embedding(region_idx)
        
        combined = torch.cat([genre, bpm, energy, region], dim=-1)
        return self.combine(combined)

# Sub-genre mapping
AMAPIANO_SUBGENRES = {
    0: 'Classic Piano',
    1: 'Private School',
    2: 'Vocal Amapiano',
    3: 'Tech Amapiano',
    4: 'Groove Amapiano',
    5: 'Deep Amapiano',
    6: 'Soulful Amapiano',
    7: 'Gqom-Amapiano Fusion',
    8: 'Afro-Tech',
    9: 'Experimental'
}

REGIONS = {
    0: 'Johannesburg',
    1: 'Pretoria',
    2: 'Durban',
    3: 'Cape Town'
}

conditioner = AmapianoConditioner().to(device)
test_condition = conditioner(
    torch.tensor([0]).to(device),  # Classic Piano
    torch.tensor([3]).to(device),  # 115 BPM bin
    torch.tensor([1]).to(device),  # Medium energy
    torch.tensor([0]).to(device)   # Johannesburg
)
print(f"Condition embedding shape: {test_condition.shape}")

## 4. Complete Generation Pipeline

In [ ]:
class AmapianoGenerator(nn.Module):
    """Complete Amapiano music generation model."""
    
    def __init__(self, vocab_size=1024, d_model=512):
        super().__init__()
        
        self.tokenizer = AudioTokenizer(vocab_size, d_model // 4)
        self.conditioner = AmapianoConditioner(d_model)
        self.transformer = AudioTransformer(vocab_size, d_model)
    
    def encode_audio(self, audio):
        return self.tokenizer.encode(audio)
    
    def decode_tokens(self, tokens):
        return self.tokenizer.decode(tokens)
    
    def forward(self, audio, genre_idx, bpm_idx, energy_idx, region_idx):
        # Encode audio to tokens
        _, tokenizer_loss, tokens = self.tokenizer(audio)
        
        # Get conditioning
        condition = self.conditioner(genre_idx, bpm_idx, energy_idx, region_idx)
        
        # Transformer forward (teacher forcing)
        logits = self.transformer(tokens, condition)
        
        # Calculate loss
        lm_loss = F.cross_entropy(
            logits[:, :-1].reshape(-1, logits.size(-1)),
            tokens[:, 1:].reshape(-1)
        )
        
        total_loss = tokenizer_loss + lm_loss
        
        return logits, total_loss
    
    @torch.no_grad()
    def generate(self, genre_idx, bpm_idx, energy_idx, region_idx, 
                 duration_seconds=10, temperature=0.9, top_k=50):
        """Generate Amapiano audio."""
        self.eval()
        
        # Calculate number of tokens needed
        # Assuming ~69 tokens per second at 22050 Hz with 320x downsampling
        num_tokens = int(duration_seconds * 22050 / 320)
        
        # Get conditioning
        condition = self.conditioner(genre_idx, bpm_idx, energy_idx, region_idx)
        
        # Start with random token
        start_tokens = torch.randint(0, self.transformer.vocab_size, (1, 1)).to(next(self.parameters()).device)
        
        # Generate tokens
        tokens = start_tokens
        for _ in tqdm(range(num_tokens), desc='Generating tokens'):
            logits = self.transformer(tokens, condition)
            logits = logits[:, -1, :] / temperature
            
            if top_k > 0:
                indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
                logits[indices_to_remove] = float('-inf')
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            tokens = torch.cat([tokens, next_token], dim=1)
        
        # Decode tokens to audio
        audio = self.decode_tokens(tokens[:, 1:])  # Skip start token
        
        return audio, tokens

# Initialize model
generator = AmapianoGenerator().to(device)
print(f"Total parameters: {sum(p.numel() for p in generator.parameters()):,}")

## 5. Training Loop

In [ ]:
def train_generator(model, dataloader, epochs=10, lr=1e-4):
    """Train the Amapiano generator."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    
    history = []
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        pbar = tqdm(dataloader, desc=f'Epoch {epoch + 1}/{epochs}')
        for batch in pbar:
            audio = batch['audio'].to(device)
            genre = batch['genre'].to(device)
            bpm = batch['bpm'].to(device)
            energy = batch['energy'].to(device)
            region = batch['region'].to(device)
            
            optimizer.zero_grad()
            _, loss = model(audio, genre, bpm, energy, region)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})
        
        scheduler.step()
        avg_loss = total_loss / len(dataloader)
        history.append(avg_loss)
        print(f'Epoch {epoch + 1}: Avg Loss = {avg_loss:.4f}')
    
    return history

# Example usage (with real data)
# history = train_generator(generator, train_dataloader, epochs=50)

## 6. Generation Example

In [ ]:
# Generate Amapiano sample (demo)
print("Generating Amapiano sample...")
print(f"Style: {AMAPIANO_SUBGENRES[0]} from {REGIONS[0]}")
print(f"BPM: ~115 | Energy: Medium")

# Note: This requires a trained model
# audio, tokens = generator.generate(
#     genre_idx=torch.tensor([0]).to(device),
#     bpm_idx=torch.tensor([3]).to(device),
#     energy_idx=torch.tensor([1]).to(device),
#     region_idx=torch.tensor([0]).to(device),
#     duration_seconds=10,
#     temperature=0.9
# )

print("\n(Train the model first to generate actual audio)")

## 7. Export Model

In [ ]:
def save_checkpoint(model, optimizer, epoch, path='checkpoint.pt'):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, path)
    print(f"Checkpoint saved to {path}")

def load_checkpoint(model, optimizer, path='checkpoint.pt'):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint['epoch']

# Save model for later use
# save_checkpoint(generator, optimizer, epoch, 'amapiano_generator.pt')